In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True" # Магическая экономия памяти

In [ ]:
! pip install -q bitsandbytes transformers peft datasets trl accelerate evaluate rouge_score > _

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import evaluate
import numpy as np

# === Конфигурация ===
model_name = "Qwen/Qwen3-4B-Instruct-2507"
output_dir = "./qwen3_lora_summarization"
dataset_name = "ccdv/arxiv-summarization"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# === Модель ===
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.bfloat16,
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

In [ ]:
dataset = load_dataset(dataset_name, "section", streaming=True)
train_iter = dataset["train"].shuffle(seed=42).take(1000)

from torch.utils.data import IterableDataset

class StreamingDataset(IterableDataset):
    def __init__(self, generator_fn):
        self.generator_fn = generator_fn
    def __iter__(self):
        return self.generator_fn()

def preprocess_stream(examples_iter):
    for ex in examples_iter:
        messages = [
            {"role": "user", "content": f"Summarize this research paper:\n\n{ex['article']}"},
            {"role": "assistant", "content": ex['abstract']}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        tokenized = tokenizer(text, truncation=True, max_length=512)
        tokenized["labels"] = tokenized["input_ids"].copy()
        yield {k: torch.tensor(v, dtype=torch.long) for k, v in tokenized.items()}

train_iter = dataset["train"].shuffle(seed=42).take(1000)
eval_iter = dataset["validation"].take(100)

train_dataset = StreamingDataset(lambda: preprocess_stream(train_iter))
eval_dataset = StreamingDataset(lambda: preprocess_stream(eval_iter))


data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

README.md: 0.00B [00:00, ?B/s]

In [ ]:
# === 5. Метрики (ROUGE) ===
rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    preds = np.argmax(preds, axis=-1)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    results = rouge.compute(predictions=decoded_preds, references=decoded_labels)
    return {k: round(v * 100, 2) for k, v in results.items()}


# === 6. Аргументы тренировки ===
args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-4,
    max_steps=125,  # для стримингового датасета
    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none"
)


In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=None,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
model.save_pretrained(f"{output_dir}/adapter")
tokenizer.save_pretrained(f"{output_dir}/adapter")

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss


('./qwen3_lora_summarization/adapter/tokenizer_config.json',
 './qwen3_lora_summarization/adapter/special_tokens_map.json',
 './qwen3_lora_summarization/adapter/chat_template.jinja',
 './qwen3_lora_summarization/adapter/vocab.json',
 './qwen3_lora_summarization/adapter/merges.txt',
 './qwen3_lora_summarization/adapter/added_tokens.json',
 './qwen3_lora_summarization/adapter/tokenizer.json')

In [ ]:
eval_args = TrainingArguments(
    output_dir=output_dir,
    do_train=False,
    do_eval=True,
    per_device_eval_batch_size=8,  # можно увеличить, так как нет обучения
    prediction_loss_only=False,
)

# Создаем трейнер для валидации
eval_trainer = Trainer(
    model=model,
    args=eval_args,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Проводим валидацию
eval_results = eval_trainer.evaluate()
print("\nFinal Evaluation Results:")
for key, value in eval_results.items():
    print(f"{key}: {value}")